# Tutorial: Milestone 2 Verification (Sessions, ViewState, Selectors, Hashing)

Audience:
- Contributors validating Milestone 2 parity between Python and Rust runtimes.

Prerequisites:
- Run from the repository root (`/Users/austin/GitHub/lucida`).
- `uv`, `cargo`, and test dependencies are available.

Expected outcomes:
- Rust Milestone 2 daemon tests pass.
- Python oracle view-state tests remain green.
- Milestone 2 parity corpus passes on both Python and Rust backends.
- Fixture contains the expected Milestone 2 case names.


## Outline

1. Setup helper utilities
2. Run Rust Milestone 2 integration tests
3. Run Python oracle tests for view-state behavior
4. Run Milestone 2 parity test on Python and Rust backends
5. Check fixture case coverage


In [1]:
from __future__ import annotations

import json
import os
import subprocess
from pathlib import Path

cwd = Path.cwd()
if (cwd / 'MIGRATION.md').exists():
    REPO_ROOT = cwd
elif (cwd.parent.parent / 'MIGRATION.md').exists():
    REPO_ROOT = cwd.parent.parent
else:
    raise AssertionError('Could not locate repository root containing MIGRATION.md')

def run(cmd: str, *, extra_env: dict[str, str] | None = None) -> str:
    env = dict(os.environ)
    if extra_env:
        env.update(extra_env)

    print(f"$ {cmd}")
    completed = subprocess.run(
        cmd,
        shell=True,
        cwd=REPO_ROOT,
        env=env,
        text=True,
        capture_output=True,
    )

    if completed.stdout:
        print(completed.stdout)
    if completed.returncode != 0:
        if completed.stderr:
            print(completed.stderr)
        raise AssertionError(f"Command failed ({completed.returncode}): {cmd}")
    return completed.stdout

print(f"Repository root: {REPO_ROOT}")


Repository root: /Users/austin/GitHub/lucida


## Step 1 - Rust Milestone 2 integration tests

This validates the new Rust routes and selector/hash behavior for Milestone 2.
Expectation: test process exits successfully and reports all tests passing.


In [2]:
rust_out = run('cargo test -p lucida-daemon --test view_state')
assert 'test result: ok.' in rust_out


$ cargo test -p lucida-daemon --test view_state

running 6 tests
test view_create_unknown_dataset_error ... ok
test session_create_returns_unique_ids ... ok
test view_end_to_end_create_update_get ... ok
test view_create_unsupported_mode_error ... ok
test view_update_invalid_patch_error ... ok
test selector_clamp_and_strict_modes ... ok

test result: ok. 6 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.02s




## Step 2 - Python oracle test suite for view-state behavior

This confirms the canonical Python behavior has not regressed while Rust is being migrated.
Expectation: all listed test modules pass.


In [3]:
python_oracle_out = run(
    'uv run pytest '
    'tests/test_view_state_service.py '
    'tests/test_view_state_endpoints.py '
    'tests/test_view_state_client.py '
    'tests/test_view_state_cli.py -q'
)
assert 'passed' in python_oracle_out


$ uv run pytest tests/test_view_state_service.py tests/test_view_state_endpoints.py tests/test_view_state_client.py tests/test_view_state_cli.py -q
..............                                                           [100%]
14 passed in 0.36s



## Step 3 - Milestone 2 parity test on both runtimes

First run against Python (fixture oracle), then against Rust.
Expectation: both runs report `1 passed`.


In [4]:
parity_python_out = run(
    'uv run pytest tests/test_milestone2_viewstate_parity.py -q',
    extra_env={'LUCIDA_TEST_BACKEND': 'python'},
)
assert '1 passed' in parity_python_out

parity_rust_out = run(
    'uv run pytest tests/test_milestone2_viewstate_parity.py -q',
    extra_env={'LUCIDA_TEST_BACKEND': 'rust'},
)
assert '1 passed' in parity_rust_out


$ uv run pytest tests/test_milestone2_viewstate_parity.py -q
.                                                                        [100%]
1 passed in 0.13s

$ uv run pytest tests/test_milestone2_viewstate_parity.py -q
.                                                                        [100%]
1 passed in 0.80s



## Step 4 - Fixture coverage check

This verifies the Milestone 2 fixture includes expected scenario names.


In [5]:
fixture_path = REPO_ROOT / 'tests' / 'parity' / 'fixtures' / 'milestone2' / 'viewstate_corpus.json'
fixture_payload = json.loads(fixture_path.read_text(encoding='utf-8'))
case_names = [case['name'] for case in fixture_payload['cases']]

expected_names = {
    'session_create_success',
    'dataset_open_with_session_success',
    'view_create_success',
    'view_get_success',
    'view_update_success',
    'view_update_index_clamped_success',
    'view_update_range_clamped_success',
    'view_update_set_clamped_success',
    'view_update_selector_index_out_of_bounds_error',
    'view_update_selector_range_out_of_bounds_error',
    'view_update_selector_set_out_of_bounds_error',
    'view_update_invalid_patch_error',
    'view_create_unknown_dataset_error',
    'view_create_unsupported_mode_error',
    'view_get_wrong_session_error',
    'view_get_unknown_session_error',
}

missing = sorted(expected_names - set(case_names))
assert not missing, f'Missing fixture cases: {missing}'

{'case_count': len(case_names), 'first_five_cases': case_names[:5]}


{'case_count': 17,
 'first_five_cases': ['session_create_success',
  'dataset_open_with_session_success',
  'view_create_success',
  'view_get_success',
  'view_update_success']}